<img src="https://res.cloudinary.com/dtizipxds/image/upload/q_auto/f_auto/v1776264397/banner_yvwajv.png" width="100%">


In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn


# Template - Anomaly Detection
Simple comparison of `z-score`, `Isolation Forest`, and `LOF` with clear plots.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
DATA_PATH = Path("../Solutions/Anomaly Detection/data.csv")
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


## 1) Prepare Features


In [ ]:
features = [c for c in df.columns if c.startswith("V")] + ["amount"]
X = df[features].copy()
X_scaled = StandardScaler().fit_transform(X)
print("Features:", len(features))


## 2) Z-Score Method


In [ ]:
z = np.abs((X - X.mean()) / X.std(ddof=0))
z_flag = (z > 3).any(axis=1).astype(int)
print("Z-score anomalies:", z_flag.sum())


## 3) Isolation Forest


In [ ]:
iso = IsolationForest(contamination=0.01, random_state=42)
iso_pred = iso.fit_predict(X_scaled)
iso_flag = (iso_pred == -1).astype(int)
print("Isolation Forest anomalies:", iso_flag.sum())


## 4) Local Outlier Factor (LOF)


In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
lof_pred = lof.fit_predict(X_scaled)
lof_flag = (lof_pred == -1).astype(int)
print("LOF anomalies:", lof_flag.sum())


## 5) Visual Comparison


In [ ]:
viz = df[["amount", "time_s"]].copy()
viz["zscore"] = z_flag
viz["isolation_forest"] = iso_flag
viz["lof"] = lof_flag

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
for ax, col, title in zip(axes, ["zscore", "isolation_forest", "lof"], ["Z-score", "Isolation Forest", "LOF"]):
    sns.scatterplot(data=viz.sample(n=min(4000, len(viz)), random_state=42), x="time_s", y="amount", hue=col, palette=["#9aa5b1", "#d7263d"], s=20, alpha=0.8, ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.show()
